# شام — مسار جمع وترميز الفيديو (بلا حاجة لـ GPU)

**فرق مهم عن دفتري الصوت والصورة الشقيقين، ذُكر بصراحة كي لا يُفهم خطأً:** الفيديو **لا يملك أداة ترميز (tokenizer) خاصة به إطلاقاً** — هو ببساطة سلسلة من رموز الصورة نفسها، إطاراً إطاراً (`video_tokenizer.py`). لذلك هذا الدفتر **لا يدرّب شيئاً جديداً** — وظيفته فقط: جمع فيديوهات حقيقية، استخراج إطارات حقيقية منها عبر ffmpeg، ثم تحويلها لتسلسل رموز عبر أداة ترميز الصورة **المدرَّبة مسبقاً** (من دفتر `sham_image_tokenizer_track`)، ونشر النتيجة كمجموعة بيانات نصية-رمزية جاهزة لمرحلة تدريب الفيديو القادمة.

**نطاق حقيقي يجب معرفته بصراحة:** لا يوجد مصدر فيديو عربي مكافئ متاح بسهولة بنفس حجم/جاهزية `UCF101` — يستخدم هذا الدفتر مجموعة فيديو حقيقية معروفة للأبحاث (`sayakpaul/ucf101-subset`)، لأن المطلوب هنا هو تنوّع حركة/مشاهد حقيقية لتعلّم البنية الزمنية، لا لغة التعليق (لا توجد تعليقات نصية تُستخدَم في هذه المرحلة أصلاً).

## قبل "Save Version → Save & Run All":
1. **فعّل الإنترنت** من Settings (لا حاجة لـ GPU).
2. تأكد من وجود نفس أسرار Kaggle: `GITHUB_TOKEN`، `KAGGLE_USERNAME`، `KAGGLE_KEY`.
3. **شغّلي «مسار ترميز الصورة» أولاً** حتى ينشر `sham-image-tokenizer-checkpoint` — إلزامي (بلا أداة ترميز مدرّبة لا معنى لرموز الفيديو)، لكن لا حاجة لإرفاقه يدوياً: يُجلب تلقائياً.
4. **من التشغيل الثاني فصاعداً**: أضف نتاج هذا الدفتر نفسه أيضاً كمدخل ليكمل تجميع الفيديوهات الجديدة فوق ما سبق.
5. استخدم **Save Version → Save & Run All (Commit)** دائماً، ويمكن جدولته للتشغيل التلقائي (Schedule this notebook to run) — بلا معالج رسومي، فلا حصة أسبوعية تحدّه.


> **الاستئناف تلقائي بالكامل (2026-09-24):** لا حاجة لأي «Add Input». في كل تشغيل يجلب الدفتر آخر نسخة من مجموعة بياناته وأداة ترميز الصورة عبر Kaggle API (`sham_inputs.py`) إن لم تكن مرفقة، ويكمل منها. إن كانت مرفقة يدوياً تُستخدم كما هي.

### 1) سحب الكود الحقيقي من GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py"))
sys.path.insert(0, CODE_DIR)
print("كود شام الحقيقي جاهز في:", CODE_DIR)


### 2) تثبيت المكتبات الإضافية

In [ ]:
try:
    import datasets
    print(f"مكتبة datasets متوفرة مسبقاً (نسخة {datasets.__version__}).")
except ImportError:
    subprocess.run(["pip", "install", "-q", "datasets"], check=True)
    print("تم تثبيت مكتبة datasets.")

import imageio_ffmpeg
print("ffmpeg الحقيقي جاهز عبر imageio-ffmpeg في:", imageio_ffmpeg.get_ffmpeg_exe())


### 3) تحميل أداة ترميز الصورة المُدرَّبة مسبقاً (إلزامي)

بعكس دفاتر الصوت/الصورة، هذا الدفتر **لا يبني أداة ترميز جديدة أبداً** — فقط يستخدم الموجودة فعلاً.

In [ ]:
from pathlib import Path
from tokenizer_select import select_pretrained_tokenizer
from sham_inputs import fetch_dataset, rglob_inputs

# بلا «Add Input»: أداة ترميز الصورة المدرَّبة + ما جُمع من فيديو سابقاً تُجلب تلقائياً.
fetch_dataset("sham-image-tokenizer-checkpoint")
fetch_dataset("sham-video-corpus")

# أداة ترميز الصورة: الأكثر تدريباً بين المدخلات، مع إصلاح غير المتوافقة وقبولها.
print("البحث عن أداة ترميز الصورة:")
_picked = select_pretrained_tokenizer("image")
assert _picked, (
    "لم يُعثر على أي أداة ترميز صورة ضمن /kaggle/input — "
    "شغّلي «مسار ترميز الصورة» أولاً حتى ينشر sham-image-tokenizer-checkpoint (يُجلب هنا تلقائياً). "
    "لا يمكن إنتاج رموز فيديو ذات معنى بلا أداة ترميز صورة مُدرَّبة فعلياً."
)
image_tokenizer, _, _image_tok_path = _picked
image_tokenizer.eval()
print(f"أداة ترميز الصورة محمّلة من: {_image_tok_path}")
print(f"حجم الصورة المتوقَّع: {image_tokenizer.cfg.image_size}x{image_tokenizer.cfg.image_size}، "
      f"عدد رموز كل صورة: {image_tokenizer.cfg.tokens_per_image}")

### 4) استئناف تقدّم جمع الفيديو من التشغيل السابق (إن وُجد)

In [ ]:
import json as _json

# الأكبر (أكثر تسلسلات) لا الأحدث تاريخاً — تواريخ الملفات داخل /kaggle/input غير موثوقة
def _lines(p):
    with open(p, encoding="utf-8") as fh:
        return sum(1 for l in fh if l.strip())
previous_video_corpus = sorted(rglob_inputs("video_corpus.jsonl"), key=_lines)
previous_video_progress = sorted(rglob_inputs("video_corpus_progress.json"),
                                 key=lambda p: _json.loads(p.read_text()).get("videos_consumed", 0))

existing_sequences = []
videos_consumed = 0
if previous_video_corpus:
    with open(previous_video_corpus[-1], encoding="utf-8") as f:
        existing_sequences = [line for line in f if line.strip()]
    if previous_video_progress:
        videos_consumed = _json.loads(previous_video_progress[-1].read_text()).get("videos_consumed", 0)
    print(f"استؤنف من: {previous_video_corpus[-1]} ({len(existing_sequences):,} تسلسل رمزي سابق، "
          f"تم استهلاك {videos_consumed:,} فيديو سابقاً)")
else:
    print("لا توجد مجموعة سابقة — بدء تجميع فيديو جديد (متوقَّع فقط في أول تشغيل حقيقي).")

### 5) جمع دفعة جديدة من فيديوهات حقيقية

`skip=videos_consumed` يضمن رؤية فيديوهات **جديدة** كل تشغيل، تماماً كمبدأ الصوت والصورة.

In [ ]:
from sham_data_sources import Ledger, collect

MAX_VIDEOS = 50  # الفيديو أثقل بكثير من الصورة (تحميل + فك ترميز فعلي) -- بداية متحفظة
NUM_FRAMES = 8

# كان المصدر السابق (ucf101-subset) يحوي فيديوهين فقط فيتكرر نفس المقطعين كل تشغيل.
# الآن: مقاطع Kinetics المرخّصة بالمشاع الإبداعي فقط، مع بصمة لكل مقطع (يُستبعد المكرر
# حتى لو أعيد ترميزه) وموضع استئناف حقيقي (sham_data_sources.py).
video_ledger = Ledger.load("video")
video_manifest, collect_stats = collect(
    "video", "/kaggle/working/corpus/videos", MAX_VIDEOS, video_ledger,
    image_size=image_tokenizer.cfg.image_size, num_frames=NUM_FRAMES,
)
print(f"جاهز: {video_manifest}")


### 6) ترميز الفيديوهات الحقيقية عبر أداة ترميز الصورة المحمَّلة

لا تدريب هنا إطلاقاً (`torch.no_grad()`) -- فقط استدلال حقيقي بأداة ترميز جاهزة، ثم تخزين التسلسلات الناتجة.

In [ ]:
import torch
from PIL import Image
from video_tokenizer import encode_video

video_root = Path(video_manifest).parent
new_sequences = []
videos_this_run = 0

with open(video_manifest, encoding="utf-8") as f:
    for line in f:
        record = _json.loads(line)
        frames = [
            Image.open(video_root / fp).convert("RGB").resize((image_tokenizer.cfg.image_size, image_tokenizer.cfg.image_size))
            for fp in record["frames"]
        ]
        frame_tensors = torch.stack([
            torch.tensor(list(im.getdata()), dtype=torch.float32)
            .view(image_tokenizer.cfg.image_size, image_tokenizer.cfg.image_size, 3)
            .permute(2, 0, 1) / 127.5 - 1.0
            for im in frames
        ]).unsqueeze(0)  # (1, num_frames, 3, H, W)

        with torch.no_grad():
            sequence = encode_video(image_tokenizer, frame_tensors)
        new_sequences.append(_json.dumps({"tokens": sequence[0].tolist(), "num_frames": NUM_FRAMES}))
        videos_this_run += 1

print(f"تم ترميز {videos_this_run:,} فيديو حقيقي جديد إلى تسلسلات رمزية حقيقية "
      f"(طول كل تسلسل: {2 + NUM_FRAMES * (2 + image_tokenizer.cfg.tokens_per_image):,} رمزاً).")


### 7) حفظ ونشر مجموعة رموز الفيديو (كمجموعة بيانات Kaggle خاصة بهذا المسار)

In [ ]:
all_sequences = existing_sequences + new_sequences
final_videos_consumed = videos_consumed + videos_this_run

Path("/kaggle/working/checkpoints").mkdir(parents=True, exist_ok=True)
with open("/kaggle/working/checkpoints/video_corpus.jsonl", "w", encoding="utf-8") as f:
    for line in all_sequences:
        f.write(line if line.endswith("\n") else line + "\n")
video_ledger.save("/kaggle/working/checkpoints")
(Path("/kaggle/working/checkpoints") / "video_corpus_progress.json").write_text(
    _json.dumps({"videos_consumed": final_videos_consumed})
)
print(f"إجمالي التسلسلات الرمزية المحفوظة الآن: {len(all_sequences):,} "
      f"(إجمالي الفيديوهات المُستهلكة: {final_videos_consumed:,}).")


In [ ]:
import json as _json
import shutil as _shutil

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=False)

KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")
DATASET_SLUG = f"{KAGGLE_USERNAME}/sham-video-corpus"

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True)
_shutil.copy2("/kaggle/working/checkpoints/video_corpus.jsonl", upload_dir / "video_corpus.jsonl")
_shutil.copy2("/kaggle/working/checkpoints/video_corpus_progress.json", upload_dir / "video_corpus_progress.json")
_shutil.copy2("/kaggle/working/checkpoints/video_ledger.json", upload_dir / "video_ledger.json")

metadata = {"title": "sham-video-corpus", "id": DATASET_SLUG, "licenses": [{"name": "unknown"}]}
(upload_dir / "dataset-metadata.json").write_text(_json.dumps(metadata))

# نفس الفحص الحتمي المعتمد في كل الدفاتر الأخرى -- نسأل Kaggle مباشرة هل
# مجموعة البيانات موجودة أصلاً بدل تخمين ذلك من نص رسالة خطأ.
_list_result = subprocess.run(["kaggle", "datasets", "list", "-m", "--csv"], capture_output=True, text=True)
_dataset_exists = DATASET_SLUG in (_list_result.stdout or "")

# "-r zip" وليس "skip": القيمة الافتراضية الموثّقة لـ Kaggle CLI للمجلدات
# الفرعية هي تجاهلها بصمت -- درس مستفاد من خطأ حقيقي سابق أثّر على دفاتر
# أخرى قبل اكتشافه.
if _dataset_exists:
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", f"auto-update: {final_videos_consumed:,} videos consumed", "-r", "zip"],
        capture_output=True, text=True,
    )
else:
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(upload_dir), "-r", "zip"],
        capture_output=True, text=True,
    )
_combined = (result.stdout or "") + (result.stderr or "")

if result.returncode == 0 and "error" not in _combined.lower():
    verb = "تم النشر إلى" if _dataset_exists else "تم إنشاء"
    print(f"{verb} {DATASET_SLUG} -- التشغيل المجدول القادم سيلتقطها تلقائياً ويكمل من حيث توقفنا.")
elif "incompatible" in _combined.lower():
    print(
        "تحذير: هذه المجموعة أُنشئت سابقاً بصيغة رفع غير متوافقة. لا يوجد إصلاح على مستوى الكود لها تحديداً -- "
        "غيّر الاسم أعلاه لاسم لم يُستخدم من قبل (مثلاً أضف -v2) وأعد التشغيل. البيانات نفسها آمنة "
        "في Output هذه الجلسة بغض النظر."
    )
    print(_combined)
else:
    print("تحذير: فشل النشر -- البيانات لا تزال آمنة في Output هذه الجلسة. تأكد من صحة KAGGLE_USERNAME/KAGGLE_KEY.")
    print(_combined)
